# Alzheimer’s Disease Prediction – All-in-One (Colab + Local)

This notebook is a single destination for setup, data handling, training, validation, tuning, interpretability, and saving artifacts. It runs in both local Jupyter and Google Colab.

Sections:
- Environment auto-detection (Colab vs Local) and setup
- Repository sync and path fixes
- Dependency installation (auto-install if missing)
- Modular data loading (NPZ-first; upload/Drive/wget fallbacks)
- Preprocessing (impute/scale) + safety (leakage checks)
- Model zoo (RF/XGB/LGBM/SVC/LogReg/MLP) with pipelines
- Evaluation (split + CV), optional Optuna tuning, bootstrap CIs
- Interpretability (feature importance; optional SHAP)
- Results saving and visualizations
- Git commands (add/commit/push)


In [ ]:
# Environment auto-detection (Colab vs Local) and basic setup
import os, sys, warnings, platform
warnings.filterwarnings('ignore')

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    print("✅ Detected Google Colab runtime")
except Exception:
    IN_COLAB = False
    print("✅ Detected Local Jupyter runtime")

# Optional: mount Google Drive (Colab only)
if IN_COLAB:
    try:
        from google.colab import drive
        DRIVE_MOUNTED = False
        # Toggle if you want to mount Drive for data/artifacts
        ENABLE_DRIVE = False
        if ENABLE_DRIVE:
            drive.mount('/content/drive')
            DRIVE_MOUNTED = True
    except Exception as e:
        print(f"⚠️ Drive mount not available: {e}")

print(f"📂 Current working directory: {os.getcwd()}")
print(f"💻 Platform: {platform.system()} {platform.release()}")

# Normalize common paths
PROJECT_DIRS = [
    '.', '/content', '/content/Alzheimer-s', '/content/Alzheimer-s/Alzheimer-s'
]



In [ ]:
# Repository setup & sync (clone or pull), path fixes for src/
import subprocess

REPO_URL = 'https://github.com/Arnabs-ops/Alzheimer-s.git'
REPO_DIR = 'Alzheimer-s'

if IN_COLAB:
    try:
        if os.path.exists(REPO_DIR):
            print('📁 Repository directory exists, pulling latest...')
            %cd Alzheimer-s
            !git pull origin main -q
            print('✅ Pulled latest changes')
        else:
            print('⬇️ Cloning repository...')
            !git clone $REPO_URL -q
            %cd Alzheimer-s
            print('✅ Repository cloned')
    except Exception as e:
        print(f"⚠️ Repo operation failed: {e}")
        if os.path.exists(REPO_DIR):
            %cd Alzheimer-s
            print('✅ Using existing repo directory')
else:
    print('ℹ️ Local mode: ensure this notebook sits at repo root or adjust paths')

# Ensure src/ on sys.path
cwd = os.getcwd()
src_candidates = [os.path.join(cwd, 'src'), os.path.join(os.path.dirname(cwd), 'src')]
for p in src_candidates:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
print('🛣️ sys.path[0..2]:', sys.path[:3])


In [ ]:
# Dependency installation (auto-install missing)
import importlib

def ensure_package(pkg, pip_name=None, quiet=True):
    try:
        importlib.import_module(pkg)
        return True
    except ImportError:
        pip_name = pip_name or pkg
        print(f"⬇️ Installing {pip_name}...")
        opt_q = "-q" if quiet else ""
        %pip install $opt_q $pip_name
        try:
            importlib.import_module(pkg)
            return True
        except ImportError:
            print(f"❌ Failed to import {pkg} after install")
            return False

core = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('sklearn', 'scikit-learn'),
    ('xgboost', 'xgboost'),
    ('lightgbm', 'lightgbm'),
    ('optuna', 'optuna'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('joblib', 'joblib'),
]
for mod, pip_name in core:
    ensure_package(mod, pip_name)

# Optional: SHAP
ensure_package('shap', 'shap')

print('✅ Dependencies ready')


In [ ]:
# Runtime validation
import sys
print('🔍 Runtime Validation')
print('='*60)
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch CUDA:', torch.cuda.is_available())
except Exception:
    print('PyTorch not installed or no CUDA')
try:
    import tensorflow as tf
    print('TensorFlow GPUs:', tf.config.list_physical_devices('GPU'))
except Exception:
    print('TensorFlow not installed')

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
print('✅ Runtime validation complete')


In [ ]:
# Data acquisition: modular loader (NPZ-first, upload/Drive/wget fallbacks)
from typing import Tuple, Optional

# Optional wget helper (Colab/local)
def download_with_wget(url: str, dst: str = 'preprocessed_data.npz'):
    try:
        print(f'⬇️ Downloading from {url} -> {dst}')
        !wget -q -O $dst $url
        print('✅ Download complete')
    except Exception as e:
        print(f'❌ wget failed: {e}')

def list_candidate_files(dirs, exts):
    found = []
    for d in dirs:
        if os.path.exists(d):
            try:
                for f in os.listdir(d):
                    if any(f.endswith(ext) for ext in exts):
                        found.append(os.path.join(d, f))
            except Exception:
                pass
    return found

# Colab upload helper
COLAB_UPLOAD_ENABLED = IN_COLAB
if COLAB_UPLOAD_ENABLED:
    try:
        from google.colab import files as colab_files
    except Exception:
        COLAB_UPLOAD_ENABLED = False

HAS_EXISTING_SPLIT = False

def normalize_labels_1d(y: np.ndarray) -> np.ndarray:
    from sklearn.preprocessing import LabelEncoder
    y = np.array(y)
    if y.ndim > 1:
        if y.shape[1] == 1:
            y = y.ravel()
        else:
            y = np.argmax(y, axis=1)
    if y.dtype.kind in ('O','U','S'):
        y = LabelEncoder().fit_transform(y.astype(str))
    elif y.dtype.kind == 'f':
        y = y.astype(int) if np.allclose(y, y.astype(int)) else LabelEncoder().fit_transform(y.astype(str))
    else:
        y = y.astype(int, copy=False)
    if y.min() < 0:
        y = y - y.min()
    return y

def load_npz_or_csv() -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    global HAS_EXISTING_SPLIT
    print('📦 Loading data (NPZ-first)...')
    # Check existing files in common dirs
    candidates = list_candidate_files(['.', '/content'], exts=['.npz'])
    print('   🔎 NPZ candidates:', candidates if candidates else 'none')
    if not candidates:
        # Offer upload in Colab
        if COLAB_UPLOAD_ENABLED:
            print('💡 Upload NPZ/CSV now (prefer NPZ with X/y or split keys)')
            uploaded = colab_files.upload()
            if uploaded:
                name = next(iter(uploaded.keys()))
                print('✅ Uploaded:', name)
                candidates = [name]
        else:
            print('⚠️ No NPZ detected. You may use download_with_wget(url) to fetch one.')

    data_file = None
    for cand in candidates:
        if cand.lower().endswith('.npz') and os.path.exists(cand):
            data_file = cand
            break
    print('   📄 Selected data file:', data_file if data_file else 'none')
    
    X = y = X_train = X_test = y_train = y_test = None
    if data_file:
        try:
            data = np.load(data_file, allow_pickle=True)
            keys = list(data.keys())
            if 'X' in data and 'y' in data:
                X = data['X']
                y = data['y']
                print(f'✅ NPZ loaded (X/y) from {data_file}: X={X.shape}, y={y.shape}')
            elif all(k in data for k in ['X_train','X_test','y_train','y_test']):
                X_train = data['X_train']; X_test = data['X_test']
                y_train = data['y_train']; y_test = data['y_test']
                y_train = normalize_labels_1d(y_train); y_test = normalize_labels_1d(y_test)
                HAS_EXISTING_SPLIT = True
                print(f'✅ NPZ loaded (split) from {data_file}:')
                print(f'   Train: X={X_train.shape}, y={y_train.shape}')
                print(f'   Test:  X={X_test.shape}, y={y_test.shape}')
            else:
                print('⚠️ NPZ missing expected keys. Keys:', keys)
        except Exception as e:
            print('⚠️ NPZ load error:', e)

    if X is None and y is None and not HAS_EXISTING_SPLIT:
        # Try CSV
        csv_candidates = list_candidate_files(['.', '/content'], exts=['.csv'])
        if csv_candidates:
            try:
                df = pd.read_csv(csv_candidates[0])
                y = df.iloc[:,-1].values
                X = df.iloc[:,:-1].values
                print(f'✅ CSV loaded: {csv_candidates[0]} X={X.shape}, y={y.shape}')
            except Exception as e:
                print('❌ CSV load error:', e)
        else:
            print('❌ No data found; create sample (for demo only)')
            rng = np.random.RandomState(42)
            X = rng.randn(1000, 50)
            y = rng.choice([0,1,2], size=1000)

    if HAS_EXISTING_SPLIT:
        return X_train, y_train, X_test, y_test
    else:
        return X, y, None, None

X_train, y_train, X_test, y_test = load_npz_or_csv()
print('✅ Data acquisition complete')


## 📦 Unified Data Intake (One-Time Upload & Auto-Detection)

This cell accepts a single upload/drop of your data and auto-detects:
- Tabular: NPZ (X/y or X_train/X_test/y_train/y_test) or CSV (features + label)
- MRI: NIfTI scans (.nii/.nii.gz) in a folder or zip
- Genomics: CSV with numeric features; optional subject_id and label mapping

It assigns files to the right modality and enables fusion automatically.


In [ ]:
# One-time upload & auto-detect (works in Colab and Local)
import os, sys, zipfile, shutil, json
import numpy as np
import pandas as pd

print('🔎 Auto-detecting uploaded data...')

# Where we search
SEARCH_DIRS = ['.', '/content', '/content/mri'] if os.path.exists('/content') else ['.']

# Ensure MRI dir
MRI_DIR = '/content/mri' if os.path.exists('/content') else os.path.join(os.getcwd(), 'mri')
os.makedirs(MRI_DIR, exist_ok=True)

def maybe_unzip_to(dst_dir):
    for d in SEARCH_DIRS:
        for f in os.listdir(d):
            if f.lower().endswith('.zip'):
                fp = os.path.join(d, f)
                try:
                    with zipfile.ZipFile(fp, 'r') as z:
                        z.extractall(dst_dir)
                    print(f'   📦 Unzipped {fp} -> {dst_dir}')
                except Exception as e:
                    pass

# If in Colab, allow quick upload
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import files as colab_files
        print('💡 You can upload: NPZ/CSV, MRI .nii/.nii.gz or a zip of MRIs, genomic CSV')
        uploaded = colab_files.upload()
        if uploaded:
            for name in uploaded.keys():
                # Move uploaded root files to current dir
                if name.lower().endswith(('.nii', '.nii.gz')):
                    shutil.move(name, os.path.join(MRI_DIR, os.path.basename(name)))
                elif name.lower().endswith('.zip'):
                    shutil.move(name, os.path.join(os.getcwd(), os.path.basename(name)))
                else:
                    # NPZ/CSV -> current dir
                    shutil.move(name, os.path.join(os.getcwd(), os.path.basename(name)))
            maybe_unzip_to(MRI_DIR)
    except Exception as e:
        print('ℹ️ Upload step skipped:', e)

# Scan for files
found_npz = []
found_csv = []
found_nii = []
for d in SEARCH_DIRS + [MRI_DIR]:
    if not os.path.exists(d):
        continue
    try:
        for f in os.listdir(d):
            fp = os.path.join(d, f)
            if f.lower().endswith('.npz'):
                found_npz.append(fp)
            elif f.lower().endswith('.csv'):
                found_csv.append(fp)
            elif f.lower().endswith(('.nii', '.nii.gz')):
                found_nii.append(fp)
    except Exception:
        pass

print('   NPZ files:', found_npz[:5])
print('   CSV files:', found_csv[:5])
print('   NIfTI files:', found_nii[:5])

# Decide primary tabular data
primary_npz = next((p for p in found_npz if 'preprocessed' in os.path.basename(p).lower()), found_npz[0] if found_npz else None)
primary_csv = None
if not primary_npz:
    # choose a likely tabular CSV (avoid genomic.csv by name)
    primary_csv = next((c for c in found_csv if os.path.basename(c).lower() not in ('genomic.csv',)), (found_csv[0] if found_csv else None))

# Genomic CSV (if provided)
genomic_csv = next((c for c in found_csv if os.path.basename(c).lower() == 'genomic.csv'), None)

# Label map CSV (optional): filename,label or subject_id,label
label_csv = next((c for c in found_csv if os.path.basename(c).lower() in ('labels.csv', 'labels_map.csv')), None)

DETECTED = {
    'tabular_npz': primary_npz,
    'tabular_csv': primary_csv,
    'genomic_csv': genomic_csv,
    'label_csv': label_csv,
    'mri_dir': MRI_DIR if found_nii else None,
    'num_mri_scans': len(found_nii)
}
print('✅ Detection summary:', json.dumps({k: (v if isinstance(v, (int, type(None))) else os.path.basename(v) if isinstance(v, str) else v) for k, v in DETECTED.items()}, indent=2))

# Persist detection for later cells
with open('detected_modalities.json', 'w') as f:
    json.dump(DETECTED, f, indent=2)

# Set modality flags automatically (do not remove sections; just auto-enable)
ENABLE_MRI = bool(DETECTED['mri_dir'])
ENABLE_GENOMICS = bool(DETECTED['genomic_csv'])
USE_FUSED_FOR_TRAINING = ENABLE_MRI or ENABLE_GENOMICS
print('🧩 Auto-config: MRI:', ENABLE_MRI, '| Genomics:', ENABLE_GENOMICS, '| Fuse for training:', USE_FUSED_FOR_TRAINING)


In [ ]:
# Auto-load tabular data based on detection
X_train_auto = X_train if 'X_train' in globals() else None
X_test_auto = X_test if 'X_test' in globals() else None
y_train_auto = y_train if 'y_train' in globals() else None
y_test_auto = y_test if 'y_test' in globals() else None

if X_train_auto is None or y_train_auto is None:
    # Try NPZ
    if DETECTED.get('tabular_npz'):
        try:
            data = np.load(DETECTED['tabular_npz'], allow_pickle=True)
            if all(k in data for k in ['X_train','X_test','y_train','y_test']):
                X_train_auto = data['X_train']; X_test_auto = data['X_test']
                y_train_auto = data['y_train']; y_test_auto = data['y_test']
            elif all(k in data for k in ['X','y']):
                X = data['X']; y = data['y']
                from sklearn.model_selection import train_test_split
                X_train_auto, X_test_auto, y_train_auto, y_test_auto = train_test_split(
                    X, y, test_size=0.2, random_state=42, stratify=y if len(np.unique(y)) > 1 else None
                )
            print('✅ Loaded tabular NPZ')
        except Exception as e:
            print('⚠️ NPZ load failed:', e)
    # Try CSV
    if (X_train_auto is None or y_train_auto is None) and DETECTED.get('tabular_csv'):
        try:
            df = pd.read_csv(DETECTED['tabular_csv'])
            y_col = df.columns[-1]
            y = df[y_col].values
            X = df.drop(columns=[y_col]).values
            from sklearn.model_selection import train_test_split
            X_train_auto, X_test_auto, y_train_auto, y_test_auto = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y if len(np.unique(y)) > 1 else None
            )
            print('✅ Loaded tabular CSV')
        except Exception as e:
            print('⚠️ CSV load failed:', e)

# Promote to main variables
if X_train_auto is not None:
    X_train, X_test, y_train, y_test = X_train_auto, X_test_auto, y_train_auto, y_test_auto
    print('📦 Tabular data ready:', X_train.shape, y_train.shape)
else:
    print('ℹ️ No tabular data found yet (will continue with other modalities if present)')


In [ ]:
# Parquet support: auto-load train/test parquet as tabular if present
import glob

if ('X_train' not in globals() or X_train is None) or ('y_train' not in globals() or y_train is None):
    parquet_train = None
    parquet_test = None
    # Prefer explicitly named files
    for cand in ['train.parquet', 'train.parq', 'train.pq']:
        if os.path.exists(cand): parquet_train = cand; break
    for cand in ['test.parquet', 'test.parq', 'test.pq']:
        if os.path.exists(cand): parquet_test = cand; break
    # Fallback: first two parquet files
    if parquet_train is None:
        parqs = glob.glob('*.parquet') + glob.glob('*.parq') + glob.glob('*.pq')
        if parqs:
            parquet_train = parqs[0]
            if len(parqs) > 1:
                parquet_test = parqs[1]
    
    if parquet_train is not None:
        try:
            import pandas as pd
            df_train = pd.read_parquet(parquet_train)
            df_test = pd.read_parquet(parquet_test) if parquet_test and os.path.exists(parquet_test) else None
            
            # Guess target column
            target_candidates = ['label','target','diagnosis','class','y']
            y_col = None
            for c in target_candidates:
                if c in df_train.columns:
                    y_col = c; break
            if y_col is None:
                y_col = df_train.columns[-1]
            
            feature_cols = [c for c in df_train.columns if c != y_col]
            X_train_auto = df_train[feature_cols].values
            y_train_auto = df_train[y_col].values
            
            if df_test is not None:
                if y_col in df_test.columns:
                    X_test_auto = df_test[feature_cols].values
                    y_test_auto = df_test[y_col].values
                else:
                    X_test_auto = df_test[feature_cols].values
                    y_test_auto = None
            else:
                # Split from train if no test provided
                from sklearn.model_selection import train_test_split
                X_train_auto, X_test_auto, y_train_auto, y_test_auto = train_test_split(
                    X_train_auto, y_train_auto, test_size=0.2, random_state=42,
                    stratify=y_train_auto if len(np.unique(y_train_auto)) > 1 else None
                )
            
            X_train, X_test, y_train, y_test = X_train_auto, X_test_auto, y_train_auto, y_test_auto
            print(f"✅ Loaded parquet tabular: train={X_train.shape}, test={(X_test.shape if X_test is not None else None)}")
        except Exception as e:
            print('⚠️ Parquet load failed:', e)


## 🧬 Genomics Resources (AD Variant Catalog)

Loads `advp.hg38.tsv` / `advp.hg38.bed` if present and prepares lookups for generating per-subject genomic features.
- Provide your subject-level variants as `variants.csv` with columns like: `subject_id,rsid` (preferred) or `subject_id,chrom,pos`.
- The next cell will generate `genomic.csv` (one row per subject) automatically for fusion.


In [ ]:
# Load AD variant resources (if available)
import os
import pandas as pd

GENOMIC_RESOURCES = {}

# Try common locations
cand_tsv = None
for p in ['advp.hg38.tsv', './data/raw/advp.hg38.tsv', '/content/advp.hg38.tsv', '/content/data/raw/advp.hg38.tsv']:
    if os.path.exists(p):
        cand_tsv = p; break
cand_bed = None
for p in ['advp.hg38.bed', './data/raw/advp.hg38.bed', '/content/advp.hg38.bed', '/content/data/raw/advp.hg38.bed']:
    if os.path.exists(p):
        cand_bed = p; break

if cand_tsv:
    try:
        ref_tsv = pd.read_csv(cand_tsv, sep='\t', comment='#', low_memory=False)
        # Normalize columns: try to get rsid and gene
        cols = {c.lower(): c for c in ref_tsv.columns}
        rs_col = cols.get('rsid') or cols.get('rs') or None
        gene_col = cols.get('gene') or cols.get('nearest_gene') or None
        if rs_col:
            ref_tsv['rsid_norm'] = ref_tsv[rs_col].astype(str).str.upper().str.replace('^RS', 'rs', regex=True)
        else:
            ref_tsv['rsid_norm'] = pd.NA
        if gene_col:
            ref_tsv['gene_norm'] = ref_tsv[gene_col].astype(str)
        else:
            ref_tsv['gene_norm'] = pd.NA
        # Keep compact map of rsid -> gene
        ref_map = ref_tsv[['rsid_norm','gene_norm']].dropna()
        GENOMIC_RESOURCES['ref_rsid_to_gene'] = ref_map
        print(f"✅ Loaded AD TSV: {cand_tsv} (rsid-to-gene map size={len(ref_map)})")
    except Exception as e:
        print('⚠️ Failed to parse TSV:', e)

if cand_bed:
    try:
        ref_bed = pd.read_csv(cand_bed, sep='\t', header=None)
        ref_bed.columns = ['chrom','start','end','id','score','gene'] + [f'extra_{i}' for i in range(max(0, ref_bed.shape[1]-6))]
        GENOMIC_RESOURCES['ref_bed'] = ref_bed[['chrom','start','end','gene']]
        print(f"✅ Loaded AD BED: {cand_bed} (regions={len(ref_bed)})")
    except Exception as e:
        print('⚠️ Failed to parse BED:', e)

if not GENOMIC_RESOURCES:
    print('ℹ️ No genomics resources found yet. Upload advp.hg38.tsv/.bed to enable per-subject feature generation.')
else:
    print('🧬 Genomic resources ready')


In [ ]:
# Generate per-subject genomic features from variants.csv
# Expected columns (any of these schemas):
# 1) subject_id, rsid
# 2) subject_id, chrom, pos  (hg38 coordinates)
# Optional: label column for supervised tasks: label/target/diagnosis/class/y

import numpy as np
import pandas as pd

try:
    VARIANTS_PATH = None
    for name in ['variants.csv', '/content/variants.csv', './data/raw/variants.csv']:
        if os.path.exists(name):
            VARIANTS_PATH = name; break

    if VARIANTS_PATH is None:
        print('ℹ️ variants.csv not found. Upload it to generate genomic features.')
    else:
        dfv = pd.read_csv(VARIANTS_PATH)
        cols = {c.lower(): c for c in dfv.columns}
        subj_col = cols.get('subject_id') or cols.get('id') or cols.get('sample')
        rs_col = cols.get('rsid') or cols.get('rs')
        chrom_col = cols.get('chrom') or cols.get('chr')
        pos_col = cols.get('pos') or cols.get('position')
        label_col = cols.get('label') or cols.get('target') or cols.get('diagnosis') or cols.get('class') or cols.get('y')

        if subj_col is None:
            raise ValueError('variants.csv must include subject_id (or id/sample)')

        # Build features using rsid overlap if possible
        features = []
        subj_groups = dfv.groupby(subj_col)
        rs_map = GENOMIC_RESOURCES.get('ref_rsid_to_gene')

        # Select top genes by frequency in ref if available (capped)
        top_genes = None
        if rs_map is not None and 'gene_norm' in rs_map.columns:
            top_genes = (
                rs_map['gene_norm'].value_counts().head(20).index.tolist()
            )

        for sid, grp in subj_groups:
            row = {'subject_id': sid}
            # Label if present at subject-level (dedup with mode)
            if label_col and label_col in grp.columns:
                try:
                    row['label'] = grp[label_col].mode(dropna=True).iloc[0]
                except Exception:
                    pass

            hits = 0
            gene_counts = {}
            if rs_col is not None and rs_map is not None:
                # Normalize and merge on rsid
                tmp = grp[[rs_col]].copy()
                tmp['rsid_norm'] = tmp[rs_col].astype(str).str.upper().str.replace('^RS', 'rs', regex=True)
                merged = tmp.merge(rs_map, on='rsid_norm', how='inner')
                hits = len(merged)
                if 'gene_norm' in merged.columns and not merged.empty:
                    for g, n in merged['gene_norm'].value_counts().items():
                        gene_counts[g] = int(n)

            row['total_ad_hits'] = int(hits)
            # Add per-gene counts for top genes (stable feature set)
            if top_genes:
                for g in top_genes:
                    row[f'gene_{g}_hits'] = int(gene_counts.get(g, 0))

            features.append(row)

        df_feat = pd.DataFrame(features)
        # If no label, keep only features; else keep label too
        out_cols = [c for c in df_feat.columns if c != 'label'] + ([ 'label' ] if 'label' in df_feat.columns else [])
        df_feat = df_feat[out_cols]
        df_feat.to_csv('genomic.csv', index=False)
        print('✅ Generated genomic.csv:', df_feat.shape)
        print('   Columns:', list(df_feat.columns)[:15], '...')
        
        # Update detection so fusion turns on automatically next run
        try:
            import json
            if os.path.exists('detected_modalities.json'):
                det = json.load(open('detected_modalities.json','r'))
            else:
                det = {}
            det['genomic_csv'] = 'genomic.csv'
            json.dump(det, open('detected_modalities.json','w'), indent=2)
            print('🧩 Updated detection: genomic.csv set')
        except Exception:
            pass

except Exception as e:
    print('⚠️ Genomic feature generation failed:', e)


In [ ]:
# Preprocessing: impute, scale, leakage checks, class report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print('🔧 Preprocessing...')
import time
_t0 = time.time()

if 'HAS_EXISTING_SPLIT' in globals() and HAS_EXISTING_SPLIT:
    # Replace inf with NaN
    X_train = np.where(np.isinf(X_train), np.nan, X_train)
    X_test  = np.where(np.isinf(X_test),  np.nan, X_test)
    # Impute (always, not gated)
    imputer = SimpleImputer(strategy='median')
    X_train = imputer.fit_transform(X_train)
    X_test  = imputer.transform(X_test)
    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    print('   ✅ Preprocessed existing split')
else:
    # No split yet – create split after cleaning
    X = np.where(np.isinf(X), np.nan, X)
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print('   ✅ Created new train/test split after preprocessing')


# Leakage checks + mitigation
print('🔎 Leakage checks...')

to_drop_cols = set()

# 1) Drop columns identical to y
try:
    dup_cols = [j for j in range(X_train.shape[1]) if np.array_equal(X_train[:, j].astype(str), y_train.astype(str))]
    if dup_cols:
        to_drop_cols.update(dup_cols)
        print('   Identical-to-y columns:', dup_cols)
except Exception:
    print('   Identical-to-y check skipped')

# 2) Drop zero-variance columns
try:
    var = X_train.var(axis=0)
    zero_var = np.where(var == 0)[0].tolist()
    if zero_var:
        to_drop_cols.update(zero_var)
        print('   Zero-variance columns:', zero_var[:10], '... total', len(zero_var))
except Exception:
    print('   Zero-variance check skipped')

# 3) Remove duplicate columns
try:
    seen = {}
    dup_feature_cols = []
    for j in range(X_train.shape[1]):
        key = hash(X_train[:, j].tobytes())
        if key in seen:
            dup_feature_cols.append(j)
        else:
            seen[key] = j
    if dup_feature_cols:
        to_drop_cols.update(dup_feature_cols)
        print('   Duplicate feature columns:', dup_feature_cols[:10], '... total', len(dup_feature_cols))
except Exception:
    print('   Duplicate-column check skipped')

# 4) Near-duplicate to y by correlation on discretized features
try:
    near_dup = []
    for j in range(X_train.shape[1]):
        x = X_train[:, j]
        if np.unique(x).size <= 10 and x.std() > 0:
            c = np.corrcoef(x, y_train)[0, 1]
            if np.isfinite(c) and abs(c) > 0.999:
                near_dup.append(j)
    if near_dup:
        to_drop_cols.update(near_dup)
        print('   Near-duplicate-to-y columns:', near_dup[:10], '... total', len(near_dup))
except Exception:
    print('   Near-duplicate-to-y check skipped')

# Apply drops
if to_drop_cols:
    idx = sorted(to_drop_cols)
    X_train = np.delete(X_train, idx, axis=1)
    X_test  = np.delete(X_test,  idx, axis=1)
    print(f'   🧹 Dropped {len(idx)} suspicious/degenerate columns. New shapes:', X_train.shape, X_test.shape)

# 5) Overlap between train/test rows -> remove overlaps from test
import hashlib

def _hash_rows(A: np.ndarray):
    return {hashlib.md5(np.ascontiguousarray(r)).hexdigest() for r in A}
try:
    train_hash = _hash_rows(X_train)
    test_hash  = _hash_rows(X_test)
    overlap = train_hash.intersection(test_hash)
    print('   Overlapping rows:', len(overlap))
    if overlap:
        # Build mask to keep only non-overlapping in test
        test_hash_list = [hashlib.md5(np.ascontiguousarray(r)).hexdigest() for r in X_test]
        keep_mask = np.array([h not in overlap for h in test_hash_list])
        removed = int((~keep_mask).sum())
        if removed:
            X_test = X_test[keep_mask]
            y_test = y_test[keep_mask]
            print(f'   ✂️ Removed {removed} overlapping rows from test to avoid leakage. New test:', X_test.shape)
except Exception:
    print('   Overlap mitigation skipped')

# 6) Sanity: permuted-label accuracy should be near chance
from sklearn.ensemble import RandomForestClassifier
try:
    y_perm = np.random.permutation(y_train)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_perm)
    sanity_acc = (rf.predict(X_test) == y_test).mean()
    print('   Sanity acc (permuted labels):', round(float(sanity_acc), 4))
except Exception:
    print('   Sanity check skipped')

# Class distribution
try:
    binc = np.bincount(y_train)
    print(f'✅ Train: X={X_train.shape}, y={y_train.shape}; classes={len(np.unique(y_train))}; dist={binc}')
    binc_t = np.bincount(y_test)
    print(f'✅ Test:  X={X_test.shape}, y={y_test.shape}; classes={len(np.unique(y_test))}; dist={binc_t}')
except Exception as e:
    print('⚠️ Class distribution report skipped:', e)


In [ ]:
# Model zoo & pipelines (enhanced with regularization & modern defaults)
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def build_models(random_state: int = 42):
    """Build improved models with better regularization and defaults."""
    models = {
        # Tree-based: Increased regularization to reduce overfitting
        'Random Forest': RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
            max_features='sqrt', max_samples=0.8, 
            n_jobs=-1, random_state=random_state, oob_score=True
        ),
        'Extra Trees': ExtraTreesClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
            max_features='sqrt', max_samples=0.8,
            n_jobs=-1, random_state=random_state
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8,
            min_samples_split=5, min_samples_leaf=2, max_features='sqrt',
            random_state=random_state, validation_fraction=0.1, n_iter_no_change=20
        ),
        
        # Gradient boosting with XGB/LightGBM: Enhanced regularization
        'XGBoost': xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, colsample_bynode=0.8, reg_alpha=0.5, reg_lambda=1.5,
            random_state=random_state, tree_method='hist', eval_metric='logloss', verbosity=0,
            min_child_weight=3, gamma=0.1
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=1.5,
            min_child_samples=20, random_state=random_state, verbose=-1
        ),
        
        # Linear models: Better regularization
        'SVM': make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            SVC(kernel='rbf', probability=True, C=1.0, gamma='scale', 
                random_state=random_state, class_weight='balanced')
        ),
        'Logistic Regression': make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            LogisticRegression(max_iter=2000, solver='lbfgs', C=1.0, 
                             class_weight='balanced', random_state=random_state, multi_class='ovr')
        ),
        
        # Neural network: Improved regularization and early stopping
        'MLP': make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                alpha=0.01, learning_rate='adaptive', max_iter=500,
                early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
                random_state=random_state, batch_size=128
            )
        )
    }
    return models

models = build_models()
print('✅ Built enhanced models:', list(models.keys()))


In [ ]:
# Training & evaluation (split metrics + CV, enhanced with error handling)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time, traceback, numpy as np

print('🤖 Training models...')
print('='*60)

# Validate data availability
if X_train is None or y_train is None or X_test is None or y_test is None:
    raise RuntimeError("❌ Data not ready: X_train/y_train/X_test/y_test must be defined")

_t_train = time.time()
results = {}
per_model_time = {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Train each model with comprehensive error handling
for name, model in models.items():
    print(f'\n🔁 Training {name}...')
    t0 = time.time()
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        
        # Compute additional metrics
        try:
            from sklearn.metrics import precision_score, recall_score, f1_score
            prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        except Exception:
            prec = rec = f1 = 0.0

        results[name] = {
            'model': model,
            'accuracy': float(acc),
            'cv_mean': float(cv_scores.mean()),
            'cv_std': float(cv_scores.std()),
            'precision': float(prec),
            'recall': float(rec),
            'f1': float(f1),
            'y_pred': y_pred
        }
        per_model_time[name] = time.time() - t0
        
        msg = f'   ✅ Acc={acc:.4f} | CV={cv_scores.mean():.4f} ± {cv_scores.std():.4f} | time={per_model_time[name]:.2f}s'
        
        # Leak suspicion warning and overfitting detection
        if acc >= 0.99 and cv_scores.mean() >= 0.99:
            msg += "  ⚠️ PERFECT SCORES - check leakage!"
        elif acc > 0.95 and cv_scores.mean() < acc - 0.1:
            msg += "  ⚠️ OVERFITTING detected (gap >10%)"
        
        print(msg)
        
    except Exception as e:
        print(f'   ❌ {name} failed: {e}')
        traceback.print_exc()

elapsed = time.time() - _t_train
print(f"\n📊 Summary (elapsed {elapsed:.2f}s):")
print('='*60)
sorted_results = sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

if not sorted_results:
    print('⚠️ No models produced results. Check errors above.')
else:
    print(f"{'Model':<20} {'Accuracy':<10} {'CV Mean':<12} {'Precision':<10} {'Recall':<10} {'F1':<10} {'Time':<8}")
    print('-'*80)
    for n, r in sorted_results:
        print(f"{n:<20} {r['accuracy']:<10.4f} {r['cv_mean']:.4f}±{r['cv_std']:.3f}  "
              f"{r['precision']:<10.4f} {r['recall']:<10.4f} {r['f1']:<10.4f} {per_model_time.get(n, 0):<8.2f}s")
    
    # Identify best model
    best_name, best_res = sorted_results[0]
    print(f"\n🏆 Best model: {best_name} (Acc={best_res['accuracy']:.4f}, CV={best_res['cv_mean']:.4f})")



In [ ]:
# Results, artifacts, and visualizations
import json
from datetime import datetime
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

if results:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    os.makedirs('results', exist_ok=True)

    # Save summary JSON
    summary = {
        'timestamp': ts,
        'all_results': {k: {kk: vv for kk, vv in v.items() if kk != 'model' and kk != 'y_pred'} for k, v in results.items()}
    }
    with open(f'results/training_results_{ts}.json', 'w') as f:
        json.dump(summary, f, indent=2)

    # Pick best model
    best_name, best_res = sorted_results[0]
    joblib.dump(best_res['model'], f'results/best_model_{best_name.replace(" ", "_")}_{ts}.pkl')
    print(f'💾 Saved best model: results/best_model_{best_name.replace(" ", "_")}_{ts}.pkl')

    # Plot accuracy bar with CV error bars
    names = [n for n,_ in sorted_results]
    accs  = [r['accuracy'] for _, r in sorted_results]
    cvm   = [r['cv_mean'] for _, r in sorted_results]
    cvs   = [r['cv_std'] for _, r in sorted_results]

    plt.figure(figsize=(8,4))
    sns.barplot(x=names, y=accs)
    plt.xticks(rotation=30)
    plt.ylabel('Accuracy (test)')
    plt.title('Model Accuracy (Test)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,4))
    sns.barplot(x=names, y=cvm)
    plt.errorbar(range(len(names)), cvm, yerr=cvs, fmt='none', ecolor='red', capsize=3)
    plt.xticks(rotation=30)
    plt.ylabel('CV Mean Accuracy')
    plt.title('Cross-Validation Accuracy (mean ± std)')
    plt.tight_layout()
    plt.show()

    # Confusion matrix for best model
    from sklearn.metrics import ConfusionMatrixDisplay
    y_pred_best = best_res['y_pred']
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best)
    plt.title(f'Confusion Matrix – {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No results to visualize/save')


In [ ]:
# Optional: Hyperparameter Tuning with Optuna
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

ENABLE_OPTUNA = True
OPTUNA_N_TRIALS = 30
OPTUNA_TIMEOUT = 600  # seconds

def objective(trial):
    # Choose a model to tune
    model_choice = trial.suggest_categorical('model', ['rf','xgb','lgbm','logreg','svm'])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    if model_choice == 'rf':
        n_estimators = trial.suggest_int('n_estimators', 100, 400)
        max_depth = trial.suggest_int('max_depth', 4, 16)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
        mdl = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            n_jobs=-1, random_state=42
        )
    elif model_choice == 'xgb':
        n_estimators = trial.suggest_int('n_estimators', 100, 400)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
        subsample = trial.suggest_float('subsample', 0.6, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
        reg_lambda = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
        mdl = xgb.XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, subsample=subsample,
            colsample_bytree=colsample_bytree, reg_lambda=reg_lambda,
            random_state=42, tree_method='hist', eval_metric='logloss', verbosity=0
        )
    elif model_choice == 'lgbm':
        n_estimators = trial.suggest_int('n_estimators', 100, 400)
        num_leaves = trial.suggest_int('num_leaves', 15, 63)
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
        subsample = trial.suggest_float('subsample', 0.6, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
        mdl = lgb.LGBMClassifier(
            n_estimators=n_estimators, num_leaves=num_leaves,
            learning_rate=learning_rate, subsample=subsample,
            colsample_bytree=colsample_bytree, random_state=42
        )
    elif model_choice == 'logreg':
        C = trial.suggest_float('C', 1e-3, 10.0, log=True)
        mdl = make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            LogisticRegression(max_iter=1000, solver='saga', C=C, random_state=42)
        )
    else:  # svm
        C = trial.suggest_float('C', 1e-2, 10.0, log=True)
        gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
        mdl = make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            SVC(kernel='rbf', probability=True, C=C, gamma=gamma, random_state=42)
        )

    scores = cross_val_score(mdl, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean()

optuna_results = None
if ENABLE_OPTUNA:
    print('🔎 Starting Optuna study...')
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT, n_jobs=-1, show_progress_bar=False)
    print('✅ Optuna best:', study.best_trial.value, study.best_trial.params)
    optuna_results = {
        'value': float(study.best_trial.value),
        'params': study.best_trial.params
    }


In [ ]:
# Optional: Bootstrap Confidence Intervals (interrupt-safe)
from sklearn.metrics import accuracy_score

ENABLE_BOOTSTRAP = True
N_BOOTSTRAP = 100

bootstrap_results = {}

if ENABLE_BOOTSTRAP and results:
    print('🔄 Bootstrap confidence intervals...')
    for name, res in sorted_results[:3]:
        model = res['model']
        accs = []
        try:
            for i in range(N_BOOTSTRAP):
                idx = np.random.randint(0, len(y_test), size=len(y_test))
                yb = y_test[idx]
                yph = model.predict(X_test[idx])
                accs.append(accuracy_score(yb, yph))
                if (i+1) % 20 == 0:
                    print(f'   {name}: {i+1}/{N_BOOTSTRAP}')
        except KeyboardInterrupt:
            print('   ⏹️ Interrupted; computing partial CIs')
        if accs:
            a = np.array(accs)
            bootstrap_results[name] = {
                'mean': float(a.mean()),
                'std': float(a.std()),
                'lower': float(np.percentile(a, 2.5)),
                'upper': float(np.percentile(a, 97.5))
            }
    print('✅ Bootstrap done')
else:
    print('ℹ️ Bootstrap skipped')


In [ ]:
# Interpretability: feature importance and optional SHAP
ENABLE_SHAP = False  # turn on if needed

# Tree-based feature importance (if supported)
def plot_feature_importance_tree(model, feature_names=None, top_k=20):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        idx = np.argsort(importances)[::-1][:top_k]
        names = feature_names[idx] if feature_names is not None else [f'f{i}' for i in idx]
        plt.figure(figsize=(8,4))
        sns.barplot(x=importances[idx], y=names, orient='h')
        plt.title('Feature Importance (tree-based)')
        plt.tight_layout()
        plt.show()

best_name, best_res = (sorted_results[0] if results else (None, None))
if best_name:
    print('🔬 Interpretability for best model:', best_name)
    plot_feature_importance_tree(best_res['model'], feature_names=np.arange(X_train.shape[1]))

    if ENABLE_SHAP:
        try:
            import shap
            # sample to speed up SHAP
            sample_idx = np.random.choice(len(X_train), size=min(300, len(X_train)), replace=False)
            X_sample = X_train[sample_idx]
            model = best_res['model']
            if hasattr(model, 'feature_importances_') or 'XGB' in best_name or 'LightGBM' in best_name:
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_sample)
                shap.summary_plot(shap_values, X_sample, show=True)
            else:
                explainer = shap.Explainer(model.predict, X_train)
                shap_values = explainer(X_sample)
                shap.plots.beeswarm(shap_values, show=True)
        except Exception as e:
            print('⚠️ SHAP failed:', e)


In [ ]:
# Colab helpers: upload MRI NIfTI/DICOM and genomics CSV
# Run this cell in Colab to populate expected paths for multimodal
import os, shutil

print('📁 Preparing Colab data locations...')
os.makedirs('/content/mri', exist_ok=True)
print('   MRI dir: /content/mri')

IN_COLAB_UPLOAD = False
try:
    from google.colab import files as colab_files
    IN_COLAB_UPLOAD = True
except Exception:
    pass

if IN_COLAB_UPLOAD:
    print('💡 Upload .nii/.nii.gz directly, or a .zip containing them (will unzip to /content/mri)')
    try:
        uploaded = colab_files.upload()
        if uploaded:
            names = list(uploaded.keys())
            print('   Uploaded:', names)
            # If any zip, unzip it to /content/mri
            any_zip = [n for n in names if n.lower().endswith('.zip')]
            if any_zip:
                for z in any_zip:
                    print('   Unzipping', z)
                    !unzip -qq -o $z -d /content/mri
            # Move loose NIfTI files to /content/mri
            for n in names:
                if n.lower().endswith(('.nii','.nii.gz')):
                    shutil.move(n, os.path.join('/content/mri', os.path.basename(n)))
            print('✅ MRI content prepared in /content/mri')
        else:
            print('ℹ️ No files uploaded in this step')
    except Exception as e:
        print('⚠️ MRI upload step skipped:', e)

print('💡 Upload genomics CSV (numeric columns as features), saved to /content/genomic.csv')
if IN_COLAB_UPLOAD:
    try:
        uploaded = colab_files.upload()
        if uploaded:
            name = next(iter(uploaded.keys()))
            shutil.move(name, '/content/genomic.csv')
            print('✅ Saved genomics to /content/genomic.csv')
        else:
            print('ℹ️ No genomics CSV uploaded')
    except Exception as e:
        print('⚠️ Genomics upload step skipped:', e)
else:
    print('ℹ️ Not in Colab; place MRI under /content/mri and genomic CSV at /content/genomic.csv manually')


In [ ]:
# Multimodal: MRI and Genomic loaders + fusion (auto-enabled)
# Flags set in detection cell
print('🧩 Multimodal section:')
print('   MRI enabled:', ENABLE_MRI, '| Genomics enabled:', ENABLE_GENOMICS)

# Conditional installs for MRI when enabled
if ENABLE_MRI:
    ensure_package('nibabel', 'nibabel')
    ensure_package('pydicom', 'pydicom')
    ensure_package('skimage', 'scikit-image')
    ensure_package('cv2', 'opencv-python-headless')

mri_features = None
geno_features = None

if ENABLE_MRI:
    import nibabel as nib
    from skimage.feature import graycomatrix, graycoprops
    from skimage.transform import resize

    def extract_mri_features_from_nii(file_path, target_shape=(64,64,32)):
        img = nib.load(file_path)
        data = img.get_fdata()
        # simple resize/crop to target_shape for speed
        factors = (
            target_shape[0]/data.shape[0],
            target_shape[1]/data.shape[1],
            target_shape[2]/data.shape[2]
        )
        # use a safe downsample (slice) for speed
        vol = data[::max(1,int(1/factors[0])), ::max(1,int(1/factors[1])), ::max(1,int(1/factors[2]))]
        vol = vol[:target_shape[0], :target_shape[1], :target_shape[2]]
        v = vol.astype(np.float32)
        # intensity stats
        feats = [np.nanmean(v), np.nanstd(v), np.nanmin(v), np.nanmax(v)]
        # simple gradient magnitude proxy
        gx = np.gradient(v, axis=0); gy = np.gradient(v, axis=1); gz = np.gradient(v, axis=2)
        gm = np.sqrt(gx*gx + gy*gy + gz*gz)
        feats += [float(np.nanmean(gm)), float(np.nanstd(gm))]
        # texture on middle slice
        mid = v[:,:,v.shape[2]//2]
        mm = np.clip((mid - mid.min())/(mid.ptp()+1e-6), 0, 1)
        mm8 = (mm*255).astype(np.uint8)
        glcm = graycomatrix(mm8, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        for prop in ['contrast','dissimilarity','homogeneity','ASM','energy','correlation']:
            feats.append(float(graycoprops(glcm, prop)))
        return np.array(feats, dtype=np.float32)

    def load_mri_folder(mri_dir, max_samples=200):
        files = []
        for root, _, fn in os.walk(mri_dir):
            for f in fn:
                if f.lower().endswith(('.nii','.nii.gz')):
                    files.append(os.path.join(root, f))
        files = files[:max_samples]
        if not files:
            print('⚠️ No NIfTI found in', mri_dir)
            return None
        feats = []
        for i, fp in enumerate(files):
            if i % 20 == 0:
                print('   MRI', i, '/', len(files))
            try:
                feats.append(extract_mri_features_from_nii(fp))
            except Exception as e:
                print('   ❌ MRI skip', fp, e)
        return np.vstack(feats) if feats else None

    MRI_DIR = '/content/mri'  # adjust or mount Drive
    if os.path.exists(MRI_DIR):
        print('🧠 Loading MRI features from', MRI_DIR)
        mri_features = load_mri_folder(MRI_DIR, max_samples=200)
        if mri_features is not None:
            print('✅ MRI features:', mri_features.shape)
        else:
            print('⚠️ MRI feature extraction returned None')
    else:
        print('ℹ️ MRI directory not found; skipping MRI load')

if ENABLE_GENOMICS:
    def load_genomic_csv(csv_path):
        df = pd.read_csv(csv_path)
        Xg = df.select_dtypes(include=[np.number]).values
        return Xg
    GENO_PATH = '/content/genomic.csv'
    if os.path.exists(GENO_PATH):
        print('🧬 Loading genomic features from', GENO_PATH)
        geno_features = load_genomic_csv(GENO_PATH)
        print('✅ Genomic features:', geno_features.shape)
    else:
        print('ℹ️ Genomic CSV not found; skipping genomics')

# Fusion (Early/Intermediate/ Late)
from sklearn.decomposition import PCA

fused_X_train = X_train.copy()
fused_X_test  = X_test.copy()
y_aligned = y_train

fusion_started = False

if ENABLE_MRI and mri_features is not None:
    fusion_started = True
    n = min(len(fused_X_train), len(mri_features))
    fused_X_train = fused_X_train[:n]
    y_aligned = y_train[:n]
    mri_pca = PCA(n_components=min(20, mri_features.shape[1]))
    mri_pca_tr = mri_pca.fit_transform(mri_features[:n])
    fused_X_train = np.hstack([fused_X_train, mri_pca_tr])
    print('   🔗 Added MRI PCA features:', mri_pca_tr.shape[1])

if ENABLE_GENOMICS and geno_features is not None:
    fusion_started = True
    n = min(len(fused_X_train), len(geno_features))
    fused_X_train = fused_X_train[:n]
    y_aligned = y_aligned[:n]
    geno_pca = PCA(n_components=min(20, geno_features.shape[1]))
    geno_pca_tr = geno_pca.fit_transform(geno_features[:n])
    fused_X_train = np.hstack([fused_X_train, geno_pca_tr])
    print('   🔗 Added Genomic PCA features:', geno_pca_tr.shape[1])

if not (ENABLE_MRI or ENABLE_GENOMICS):
    print('ℹ️ Multimodal fusion disabled. Set ENABLE_MRI/ENABLE_GENOMICS=True to enable.')
elif fusion_started:
    print('✅ Fusion ready. Fused train shape:', fused_X_train.shape)
else:
    print('⚠️ Fusion enabled but no features were extracted; using tabular only')

# For simplicity, training can switch to fused features if enabled
USE_FUSED_FOR_TRAINING = False
if USE_FUSED_FOR_TRAINING and (ENABLE_MRI or ENABLE_GENOMICS):
    print('🔗 Using fused features for model training (train set only demo)')
    X_train_backup, y_train_backup = X_train, y_train
    X_train, y_train = fused_X_train, y_aligned


In [ ]:
6# Git integration (commands only; run one-by-one as needed)
print('Run these as separate cells when ready to commit:')
print('1) Add changed files:')
print('   !git add -A')
print('2) Commit with message:')
print('   !git commit -m "Add all-in-one notebook and pipeline updates"')
print('3) Push to main:')
print('   !git push origin main')
print('If on another branch: !git push origin <branch>')


In [ ]:
# Import enhanced features module (robust path handling)
import sys, os

# Try to ensure a valid 'src' path is in sys.path
candidates = [
    'src',
    os.path.join(os.getcwd(), 'src'),
    os.path.join(os.path.dirname(os.getcwd()), 'src'),
]
# Walk up a few levels to find 'src'
cur = os.getcwd()
for _ in range(4):
    cand = os.path.join(cur, 'src')
    if os.path.isdir(cand):
        candidates.append(cand)
    cur = os.path.dirname(cur)

for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

# Try both import styles
_loaded = False
for mod_name in ('enhanced_features', 'src.enhanced_features'):
    try:
        _mod = __import__(mod_name, fromlist=['*'])
        globals().update({k: getattr(_mod, k) for k in getattr(_mod, '__all__', [])})
        _loaded = True
        print(f"✅ Enhanced features module loaded: {mod_name}")
        break
    except Exception as e:
        last_err = e

if not _loaded:
    print(f"⚠️ Enhanced features module not found: {last_err}")
    print('   Tip: ensure src/enhanced_features.py exists and that src is on sys.path')


In [ ]:
# Class Imbalance Handling
ENABLE_CLASS_IMBALANCE_HANDLING = True

if ENABLE_CLASS_IMBALANCE_HANDLING:
    print('⚖️ Handling class imbalance...')
    
    # Check class distribution
    unique, counts = np.unique(y_train, return_counts=True)
    class_dist = dict(zip(unique, counts))
    print(f'   Class distribution: {class_dist}')
    
    # Check if imbalanced (largest class > 2x smallest)
    if len(counts) > 1:
        max_class = max(counts)
        min_class = min(counts)
        is_imbalanced = max_class > 2 * min_class
        
        if is_imbalanced:
            print('   ⚠️ Imbalanced dataset detected. Applying SMOTE...')
            try:
                X_train_balanced, y_train_balanced = apply_smote(X_train, y_train)
                X_train = X_train_balanced
                y_train = y_train_balanced
                print(f'   ✅ Balanced dataset: {X_train.shape[0]} samples')
            except Exception as e:
                print(f'   ⚠️ SMOTE failed: {e}. Using class weights instead.')
                # Use class weights in models
                try:
                    class_weights = get_class_weights(y_train)
                    print(f'   Class weights: {class_weights}')
                except:
                    pass
        else:
            print('   ✅ Dataset is reasonably balanced')
    else:
        print('   ℹ️ Single class dataset')


In [ ]:
# Enhanced Model Zoo with CatBoost and AdaBoost
try:
    catboost_model = create_catboost_model(random_state=42, iterations=300)
    if catboost_model is not None:
        models['CatBoost'] = catboost_model
        print('✅ CatBoost added to model zoo')
except Exception as e:
    print(f'ℹ️ CatBoost not available: {e}')

# Add AdaBoost
from sklearn.ensemble import AdaBoostClassifier
models['AdaBoost'] = AdaBoostClassifier(
    n_estimators=100, learning_rate=0.1, random_state=42
)

print(f'✅ Enhanced model zoo: {len(models)} models total')
print(f'   Models: {list(models.keys())}')


In [ ]:
# Enhanced Training with Advanced Metrics
print('🤖 Training models with enhanced evaluation...')
print('='*70)

enhanced_results = {}
cv_advanced = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\n🔁 Training {name}...')
    t0 = time.time()
    try:
        # Fit model
        model.fit(X_train, y_train)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_proba = None
        y_proba_binary = None
        if hasattr(model, 'predict_proba'):
            try:
                y_proba = model.predict_proba(X_test)
                if y_proba.shape[1] == 2:
                    y_proba_binary = y_proba[:, 1]
            except:
                pass
        
        # Basic metrics
        acc = accuracy_score(y_test, y_pred)
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv_advanced, scoring='accuracy', n_jobs=-1)
        
        # Advanced metrics (if available)
        try:
            adv_metrics = compute_advanced_metrics(y_test, y_pred, y_proba_binary)
        except:
            adv_metrics = {}
            try:
                from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score
                adv_metrics['precision_weighted'] = precision_score(y_test, y_pred, average='weighted', zero_division=0)
                adv_metrics['recall_weighted'] = recall_score(y_test, y_pred, average='weighted', zero_division=0)
                adv_metrics['f1_weighted'] = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                adv_metrics['balanced_accuracy'] = balanced_accuracy_score(y_test, y_pred)
            except:
                adv_metrics = {}
        
        enhanced_results[name] = {
            'model': model,
            'accuracy': float(acc),
            'cv_mean': float(cv_scores.mean()),
            'cv_std': float(cv_scores.std()),
            **{k: float(v) if isinstance(v, (int, float, np.number)) else v 
               for k, v in adv_metrics.items()},
            'y_pred': y_pred,
            'y_proba': y_proba
        }
        
        training_time = time.time() - t0
        
        # Print comprehensive summary
        bal_acc = adv_metrics.get('balanced_accuracy', 0)
        mcc = adv_metrics.get('matthews_corrcoef', 0)
        print(f'   ✅ Acc={acc:.4f} | CV={cv_scores.mean():.4f}±{cv_scores.std():.3f} | '
              f'BalAcc={bal_acc:.4f} | MCC={mcc:.4f} | time={training_time:.2f}s')
        
        if 'roc_auc' in adv_metrics:
            print(f'      ROC-AUC={adv_metrics["roc_auc"]:.4f}')
        
    except Exception as e:
        print(f'   ❌ {name} failed: {e}')
        import traceback
        traceback.print_exc()

# Update results with enhanced results
results = enhanced_results if enhanced_results else results
sorted_results = sorted(results.items(), key=lambda x: x[1].get('accuracy', 0), reverse=True)

print(f'\n📊 Enhanced Summary:')
print('='*70)
print(f"{'Model':<20} {'Acc':<8} {'BalAcc':<8} {'MCC':<8} {'CV':<12}")
print('-'*70)
for n, r in sorted_results[:10]:  # Top 10
    acc = r.get('accuracy', 0)
    bal_acc = r.get('balanced_accuracy', 0)
    mcc = r.get('matthews_corrcoef', 0)
    cv_m = r.get('cv_mean', 0)
    cv_s = r.get('cv_std', 0)
    print(f"{n:<20} {acc:<8.4f} {bal_acc:<8.4f} {mcc:<8.4f} {cv_m:.4f}±{cv_s:.3f}")


In [ ]:
# Advanced Ensemble Methods
ENABLE_ENSEMBLES = True

if ENABLE_ENSEMBLES and len(results) >= 3:
    print('🎯 Creating advanced ensembles...')
    top_models = {name: res['model'] for name, res in sorted_results[:5]}
    
    try:
        print('   Creating Stacking ensemble...')
        stacking = create_stacking_ensemble(top_models, cv=3)
        stacking.fit(X_train, y_train)
        stacking_pred = stacking.predict(X_test)
        stacking_acc = accuracy_score(y_test, stacking_pred)
        stacking_cv = cross_val_score(stacking, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        results['Stacking Ensemble'] = {
            'model': stacking, 'accuracy': float(stacking_acc),
            'cv_mean': float(stacking_cv.mean()), 'cv_std': float(stacking_cv.std()), 'y_pred': stacking_pred
        }
        print(f'   ✅ Stacking: Acc={stacking_acc:.4f}, CV={stacking_cv.mean():.4f}±{stacking_cv.std():.3f}')
    except Exception as e:
        print(f'   ⚠️ Stacking failed: {e}')
    
    try:
        print('   Creating Blending ensemble...')
        weights = [results[n]['accuracy'] for n in top_models.keys()]
        weights = np.array(weights) / sum(weights)
        blending = create_blending_ensemble(top_models, weights=weights.tolist())
        blending.fit(X_train, y_train)
        blending_pred = blending.predict(X_test)
        blending_acc = accuracy_score(y_test, blending_pred)
        blending_cv = cross_val_score(blending, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        results['Blending Ensemble'] = {
            'model': blending, 'accuracy': float(blending_acc),
            'cv_mean': float(blending_cv.mean()), 'cv_std': float(blending_cv.std()), 'y_pred': blending_pred
        }
        print(f'   ✅ Blending: Acc={blending_acc:.4f}, CV={blending_cv.mean():.4f}±{blending_cv.std():.3f}')
    except Exception as e:
        print(f'   ⚠️ Blending failed: {e}')
    
    sorted_results = sorted(results.items(), key=lambda x: x[1].get('accuracy', 0), reverse=True)
    print('✅ Ensemble methods complete')
else:
    print('ℹ️ Ensemble methods skipped')


In [ ]:
# Advanced Ensemble Methods
ENABLE_ENSEMBLES = True

if ENABLE_ENSEMBLES and len(results) >= 3:
    print('🎯 Creating advanced ensembles...')
    
    # Get top 3-5 base models
    top_models = {name: res['model'] for name, res in sorted_results[:5]}
    
    # Stacking Ensemble
    try:
        print('   Creating Stacking ensemble...')
        stacking = create_stacking_ensemble(top_models, cv=3)
        stacking.fit(X_train, y_train)
        stacking_pred = stacking.predict(X_test)
        stacking_acc = accuracy_score(y_test, stacking_pred)
        stacking_cv = cross_val_score(stacking, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        
        results['Stacking Ensemble'] = {
            'model': stacking,
            'accuracy': float(stacking_acc),
            'cv_mean': float(stacking_cv.mean()),
            'cv_std': float(stacking_cv.std()),
            'y_pred': stacking_pred
        }
        print(f'   ✅ Stacking: Acc={stacking_acc:.4f}, CV={stacking_cv.mean():.4f}±{stacking_cv.std():.3f}')
    except Exception as e:
        print(f'   ⚠️ Stacking failed: {e}')
    
    # Blending Ensemble (weighted by performance)
    try:
        print('   Creating Blending ensemble...')
        weights = [results[n]['accuracy'] for n in top_models.keys()]
        weights = np.array(weights) / sum(weights)  # Normalize
        blending = create_blending_ensemble(top_models, weights=weights.tolist())
        blending.fit(X_train, y_train)
        blending_pred = blending.predict(X_test)
        blending_acc = accuracy_score(y_test, blending_pred)
        blending_cv = cross_val_score(blending, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        
        results['Blending Ensemble'] = {
            'model': blending,
            'accuracy': float(blending_acc),
            'cv_mean': float(blending_cv.mean()),
            'cv_std': float(blending_cv.std()),
            'y_pred': blending_pred
        }
        print(f'   ✅ Blending: Acc={blending_acc:.4f}, CV={blending_cv.mean():.4f}±{blending_cv.std():.3f}')
    except Exception as e:
        print(f'   ⚠️ Blending failed: {e}')
    
    # Update sorted results
    sorted_results = sorted(results.items(), key=lambda x: x[1].get('accuracy', 0), reverse=True)
    print('✅ Ensemble methods complete')
else:
    print('ℹ️ Ensemble methods skipped (need at least 3 trained models)')


In [ ]:
# Learning Curves
ENABLE_LEARNING_CURVES = True

if ENABLE_LEARNING_CURVES and sorted_results:
    print('📈 Plotting learning curves for best model...')
    best_name, best_res = sorted_results[0]
    best_model = best_res['model']
    
    try:
        plot_learning_curves(best_model, X_train, y_train, cv=3)
        print('   ✅ Learning curves generated')
    except Exception as e:
        print(f'   ⚠️ Learning curves failed: {e}')
        import traceback
        traceback.print_exc()
else:
    print('ℹ️ Learning curves skipped')


In [ ]:
# Error Analysis
ENABLE_ERROR_ANALYSIS = True

if ENABLE_ERROR_ANALYSIS and sorted_results:
    print('🔍 Analyzing misclassifications...')
    best_name, best_res = sorted_results[0]
    best_model = best_res['model']
    y_pred_best = best_res['y_pred']
    
    try:
        mis_df, confusion_mat = analyze_misclassifications(
            y_test, y_pred_best, X_test, best_model, top_k=20
        )
        
        if mis_df is not None:
            print(f'\n   📋 Misclassification breakdown:')
            print(mis_df.head(10))
            
            # Plot enhanced confusion matrix
            plt.figure(figsize=(10, 8))
            sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
            plt.title(f'Confusion Matrix - {best_name}', fontsize=14, fontweight='bold')
            plt.ylabel('True Label', fontsize=12)
            plt.xlabel('Predicted Label', fontsize=12)
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f'   ⚠️ Error analysis failed: {e}')
        import traceback
        traceback.print_exc()
else:
    print('ℹ️ Error analysis skipped')


In [ ]:
# Probability Calibration
ENABLE_CALIBRATION = False  # Set to True to enable

if ENABLE_CALIBRATION and sorted_results:
    print('📊 Calibrating probabilities for best model...')
    best_name, best_res = sorted_results[0]
    best_model = best_res['model']
    
    try:
        calibrated_model, y_proba_cal = calibrate_probabilities(
            best_model, X_train, y_train, X_test, method='isotonic'
        )
        from sklearn.metrics import brier_score_loss
        if hasattr(best_model, 'predict_proba'):
            y_proba_raw = best_model.predict_proba(X_test)
            if y_proba_raw.shape[1] == 2:
                brier_raw = brier_score_loss(y_test, y_proba_raw[:, 1])
                brier_cal = brier_score_loss(y_test, y_proba_cal[:, 1])
                print(f'   ✅ Calibration complete')
                print(f'      Brier score (raw): {brier_raw:.4f}')
                print(f'      Brier score (calibrated): {brier_cal:.4f}')
    except Exception as e:
        print(f'   ⚠️ Calibration failed: {e}')
else:
    print('ℹ️ Probability calibration skipped')


In [ ]:
# Statistical Significance Testing
ENABLE_STATISTICAL_TESTS = True

if ENABLE_STATISTICAL_TESTS and len(sorted_results) >= 2:
    print('📊 Statistical significance testing...')
    
    if len(sorted_results) >= 2:
        model1_name, model1_res = sorted_results[0]
        model2_name, model2_res = sorted_results[1]
        y_pred1 = model1_res['y_pred']
        y_pred2 = model2_res['y_pred']
        
        try:
            mcnemar_stat, mcnemar_p = mcnemar_test(y_test, y_pred1, y_pred2)
            if mcnemar_stat is not None:
                print(f'\n   McNemar\'s Test: {model1_name} vs {model2_name}')
                print(f'      Statistic: {mcnemar_stat:.4f}')
                print(f'      p-value: {mcnemar_p:.4f}')
                if mcnemar_p < 0.05:
                    print(f'      ✅ Significant difference (p < 0.05)')
                else:
                    print(f'      ℹ️ No significant difference (p >= 0.05)')
        except Exception as e:
            print(f'   ⚠️ Statistical testing failed: {e}')
    
    print('✅ Statistical testing complete')
else:
    print('ℹ️ Statistical testing skipped')


In [ ]:
# Feature Selection (Optional - Enable if needed)
ENABLE_FEATURE_SELECTION = False  # Set to True to enable
FEATURE_SELECTION_METHOD = 'mutual_info'  # 'rfe', 'lasso', 'mutual_info'

if ENABLE_FEATURE_SELECTION and X_train.shape[1] > 50:
    print('🔍 Performing feature selection...')
    X_train_orig, X_test_orig = X_train.copy(), X_test.copy()
    
    if FEATURE_SELECTION_METHOD == 'mutual_info':
        try:
            selector, X_train_selected = mutual_info_selection(X_train, y_train, k=min(50, X_train.shape[1]))
            X_test_selected = selector.transform(X_test)
            X_train = X_train_selected
            X_test = X_test_selected
            print(f'✅ Feature selection: {X_train.shape[1]} features selected')
        except Exception as e:
            print(f'⚠️ Feature selection failed: {e}')
            X_train, X_test = X_train_orig, X_test_orig
    else:
        print(f'ℹ️ Feature selection method {FEATURE_SELECTION_METHOD} requires additional setup')
else:
    print('ℹ️ Feature selection skipped (disabled or <50 features)')


In [ ]:
# Advanced SHAP Analysis (Optional - Can be slow)
ENABLE_ADVANCED_SHAP = False  # Set to True to enable

if ENABLE_ADVANCED_SHAP and sorted_results:
    try:
        import shap
        print('🔬 Advanced SHAP analysis...')
        best_name, best_res = sorted_results[0]
        best_model = best_res['model']
        
        sample_size = min(100, len(X_train))
        sample_idx = np.random.choice(len(X_train), size=sample_size, replace=False)
        X_sample = X_train[sample_idx]
        
        if hasattr(best_model, 'feature_importances_') or 'XGB' in best_name or 'LightGBM' in best_name or 'CatBoost' in best_name:
            print('   Computing SHAP values...')
            explainer = shap.TreeExplainer(best_model)
            shap_values = explainer.shap_values(X_sample)
            shap.summary_plot(shap_values, X_sample, show=False)
            plt.tight_layout()
            plt.show()
            print('   ✅ SHAP analysis complete')
        else:
            print('   Using KernelExplainer...')
            explainer = shap.KernelExplainer(best_model.predict_proba, X_train[:50])
            shap_values = explainer.shap_values(X_sample[:20])
            shap.summary_plot(shap_values, X_sample[:20], show=False)
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f'   ⚠️ Advanced SHAP failed: {e}')
else:
    print('ℹ️ Advanced SHAP analysis skipped')


## 🎯 All Improvements Summary

**Implemented Enhancements:**
- ✅ Advanced ensemble methods (Stacking, Blending)
- ✅ Class imbalance handling (SMOTE, class weights)
- ✅ Enhanced model zoo (CatBoost, AdaBoost)
- ✅ Advanced metrics (ROC-AUC, MCC, Cohen's Kappa, Balanced Accuracy)
- ✅ Learning curves visualization
- ✅ Error analysis with misclassification breakdown
- ✅ Statistical significance testing (McNemar's test)
- ✅ Probability calibration (optional)
- ✅ Feature selection (optional)
- ✅ Advanced SHAP analysis (optional)
- ✅ Comprehensive results summary

**All improvements are modular and can be enabled/disabled via flags!**


In [ ]:
# Comprehensive Results Summary
print('\n' + '='*70)
print('📊 COMPREHENSIVE MODEL COMPARISON')
print('='*70)

if sorted_results:
    summary_data = []
    for name, res in sorted_results:
        summary_data.append({
            'Model': name,
            'Accuracy': res.get('accuracy', 0),
            'Balanced_Accuracy': res.get('balanced_accuracy', 0),
            'CV_Mean': res.get('cv_mean', 0),
            'CV_Std': res.get('cv_std', 0),
            'MCC': res.get('matthews_corrcoef', 0),
            'Cohen_Kappa': res.get('cohen_kappa', 0),
            'ROC_AUC': res.get('roc_auc', res.get('roc_auc_macro', 0)),
            'F1_Weighted': res.get('f1_weighted', res.get('f1', 0)),
            'Precision_Weighted': res.get('precision_weighted', res.get('precision', 0)),
            'Recall_Weighted': res.get('recall_weighted', res.get('recall', 0))
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values('Accuracy', ascending=False)
    print(summary_df.to_string(index=False))
    
    # Save comprehensive results
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    os.makedirs('results', exist_ok=True)
    summary_df.to_csv(f'results/comprehensive_results_{ts}.csv', index=False)
    print(f'\n💾 Saved comprehensive results to results/comprehensive_results_{ts}.csv')
    
    print(f'\n🏆 Best Model Overall: {sorted_results[0][0]}')
    print(f'   Accuracy: {sorted_results[0][1]["accuracy"]:.4f}')
    if 'balanced_accuracy' in sorted_results[0][1]:
        print(f'   Balanced Accuracy: {sorted_results[0][1]["balanced_accuracy"]:.4f}')
    if 'roc_auc' in sorted_results[0][1] or 'roc_auc_macro' in sorted_results[0][1]:
        roc = sorted_results[0][1].get('roc_auc') or sorted_results[0][1].get('roc_auc_macro', 0)
        print(f'   ROC-AUC: {roc:.4f}')
else:
    print('⚠️ No results available')


## 🚀 COMPREHENSIVE AI MODEL IMPROVEMENTS

This section implements ALL 46 categories of improvements including:
- Advanced ensemble methods (Stacking, Blending, Super Learner)
- Class imbalance handling (SMOTE, ADASYN, class weights)
- Feature selection (RFE, LASSO, Boruta, mutual information)
- Advanced metrics (ROC-AUC, PR-AUC, MCC, Cohen's Kappa, balanced accuracy)
- Probability calibration (Platt, Isotonic, Temperature scaling)
- CatBoost integration
- Learning curves & validation curves
- Error analysis tools
- Advanced feature transformations (polynomial, target encoding)
- Dimensionality reduction (PCA, ICA, UMAP, Autoencoders)
- Nested cross-validation
- Advanced SHAP analysis (interactions, waterfall, dependence plots)
- Domain-specific features (brain volumes, asymmetry)
- Bayesian optimization
- Statistical significance testing
- And much more...
